# RAG Scholar Notebook

A clean, real-data walkthrough for the local RAG Scholar project.

This notebook follows the same flow as `app.py`:

1. Inspect the saved document index.
2. Audit the real chunks already stored in `rag_bm25.pkl`.
3. Test retrieval on the real `ArtOfWar.pdf` data.
4. Preview the answer format that fixes glued text and weak formatting.
5. Launch the redesigned Gradio interface.

Run cells top to bottom. The local app still owns the complete FAISS + BM25 + reranker + Groq path.

## 1. Project Paths

These files are the source of truth for the local app and the current indexed data.

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
APP_PATH = PROJECT_ROOT / "app.py"
BM25_PATH = PROJECT_ROOT / "rag_bm25.pkl"
FAISS_DIR = PROJECT_ROOT / "rag_index"

print("Project root:", PROJECT_ROOT)
print("App exists:", APP_PATH.exists())
print("BM25 chunk file exists:", BM25_PATH.exists(), BM25_PATH)
print("FAISS index exists:", FAISS_DIR.exists(), FAISS_DIR)

## 2. Current Runtime Configuration

The notebook reads configuration from the environment in the same spirit as `app.py`. Keep API keys in `.env`, not inside notebook cells.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

CONFIG = {
    "MODEL_ID": os.getenv("MODEL_ID", "llama-3.3-70b-versatile"),
    "EMBED_MODEL": os.getenv("EMBED_MODEL", "BAAI/bge-large-en-v1.5"),
    "RERANK_MODEL": os.getenv("RERANK_MODEL", "cross-encoder/ms-marco-MiniLM-L-6-v2"),
    "CHUNK_SIZE": int(os.getenv("CHUNK_SIZE", "1200")),
    "CHUNK_OVERLAP": int(os.getenv("CHUNK_OVERLAP", "300")),
    "RETRIEVAL_K": int(os.getenv("RETRIEVAL_K", "20")),
    "TOP_K": int(os.getenv("TOP_K", "5")),
    "GROQ_API_KEY_SET": bool(os.getenv("GROQ_API_KEY", "")),
}

CONFIG

## 3. Load Real Indexed Chunks

This cell uses the actual saved chunk data produced by the app. No sample or fake documents are created.

In [ ]:
import pickle
import statistics
from collections import Counter
from pathlib import Path

if not BM25_PATH.exists():
    raise FileNotFoundError("rag_bm25.pkl was not found. Upload a PDF in the app first, then rerun this notebook.")

with BM25_PATH.open("rb") as fh:
    chunks = pickle.load(fh)

sources = sorted({Path(doc.metadata.get("source", "")).name for doc in chunks})
pages = [doc.metadata.get("page") for doc in chunks if doc.metadata.get("page") is not None]
lengths = [len(doc.page_content) for doc in chunks]

print(f"Chunks loaded: {len(chunks)}")
print(f"Sources: {sources}")
print(f"Page range: {min(pages)} to {max(pages)}")
print(f"Average chunk length: {statistics.mean(lengths):.0f} characters")
print(f"Shortest / longest chunk: {min(lengths)} / {max(lengths)} characters")

## 4. Real Data Audit

A quick look at the distribution of chunks per document and a real chunk preview.

In [ ]:
source_counts = Counter(Path(doc.metadata.get("source", "")).name for doc in chunks)
page_counts = Counter(doc.metadata.get("page") for doc in chunks)

print("Chunks per source:")
for source, count in source_counts.items():
    print(f"- {source}: {count} chunks")

print()
print("Most represented pages:")
for page, count in page_counts.most_common(8):
    print(f"- page {page}: {count} chunks")

sample = chunks[0]
print()
print("Sample metadata:")
print(sample.metadata)
print()
print("Sample text:")
print(sample.page_content[:700])

## 5. RAG Flow

```mermaid
flowchart LR
  A[Uploaded PDFs] --> B[PyPDFLoader pages]
  B --> C[Recursive text chunks]
  C --> D[FAISS vector index]
  C --> E[BM25 lexical index]
  D --> F[Hybrid retrieval]
  E --> F
  F --> G[CrossEncoder reranking]
  G --> H[Prompt with cited context]
  H --> I[Groq answer in Markdown]
  I --> J[Readable Gradio chat UI]
```

The notebook cells below validate the middle of this flow using the real saved chunks.

## 6. Local BM25 Retrieval Test

This is a lightweight local retrieval check. It does not require Groq or Hugging Face downloads, so it is useful when network access is blocked.

In [ ]:
import re
from rank_bm25 import BM25Okapi

TOKEN_RE = re.compile(r"[A-Za-z0-9']+")

def tokenize(text):
    return TOKEN_RE.findall(text.lower())

corpus_tokens = [tokenize(doc.page_content) for doc in chunks]
bm25 = BM25Okapi(corpus_tokens)

print(f"BM25 ready over {len(corpus_tokens)} real chunks.")

## 7. Retrieve Real Passages for a Real Question

This mirrors the question that exposed the formatting problem in the UI.

In [ ]:
def retrieve_bm25(question, top_k=5):
    scores = bm25.get_scores(tokenize(question))
    ranked = sorted(enumerate(scores), key=lambda item: item[1], reverse=True)[:top_k]
    return [(chunks[i], float(score)) for i, score in ranked]

question = "Find definitions of the key terms."
results = retrieve_bm25(question, top_k=5)

for rank, (doc, score) in enumerate(results, start=1):
    source = Path(doc.metadata.get("source", "")).name
    page = doc.metadata.get("page", "?")
    excerpt = " ".join(doc.page_content.split())[:550]
    print(f"#{rank} score={score:.2f} | {source} p.{page}")
    print(excerpt)
    print("-" * 90)

## 8. Answer Formatting Rules Used by the App

The app now asks the model to choose a structure dynamically:

- Definitions: compact Markdown table.
- Comparisons: bullets or table.
- Processes: numbered steps.
- Evidence summaries: short sections with citations.
- Sources: Markdown list, not one long line.

The spacing bug was fixed by preserving streamed chunk whitespace instead of stripping each token.

In [ ]:
def clean_stream_chunk(text):
    """Remove model control tokens without deleting important spaces."""
    for token in ("<|assistant|>", "<|end|>", "<|user|>", "<|system|>", "<think>", "</think>"):
        text = text.replace(token, "")
    return text

# Demonstrates the old bug versus the fixed behavior.
chunks_from_stream = ["To", " find", " definitions", " of", " key", " terms"]
old_behavior = "".join(part.strip() for part in chunks_from_stream)
fixed_behavior = "".join(clean_stream_chunk(part) for part in chunks_from_stream)

print("Old behavior:  ", old_behavior)
print("Fixed behavior:", fixed_behavior)

## 9. Example of the Desired Answer Shape

This is the target format for glossary-style questions. The live app will generate the final answer from retrieved passages and Groq.

In [ ]:
example_answer = (
    "Definitions should be returned as a cited glossary, not as one long paragraph.\n\n"
    "### Key Terms\n\n"
    "| Term | Working definition | Evidence |\n"
    "|---|---|---|\n"
    "| CHENG | The direct, normal, or orthodox arrangement used when facing the enemy. | Li Ch'uan contrasts facing the enemy with lateral diversion (p.47). |\n"
    "| CH'I | The indirect, abnormal, or surprising maneuver used to secure advantage. | Chia Lin links abnormal maneuvers with victory (p.47). |\n"
    "| Fore-knowledge | Knowledge of the enemy's dispositions and intentions before action. | The text defines it as knowing what the enemy means to do (p.122). |\n\n"
    "### Sources\n"
    "- ArtOfWar.pdf p.47\n"
    "- ArtOfWar.pdf p.122\n"
)

print(example_answer)

## 10. Optional Full App Import Check

This checks that the notebook and app agree on the current indexed documents and examples.

In [ ]:
import app

print("Documents known to app:", app.doc_names)
print("Number of starter questions:", len(app.EXAMPLES))
print("First starter question:", app.EXAMPLES[0])
print("Gradio blocks:", len(app.demo.blocks))

## 11. Optional Full RAG Call

Run this only when:

- `GROQ_API_KEY` is set.
- Hugging Face embedding/reranker models are available locally or the machine has internet access.

This uses the same `respond()` path as the UI.

In [ ]:
# Optional: uncomment to test the full app path.
# gen = app.respond("Create a glossary of the key terms with short definitions and page citations.", [])
# for _ in range(3):
#     print(next(gen))

## 12. Launch the Redesigned UI

Use this when you want to interact with the full Gradio interface from the notebook.

In [ ]:
# If another server is already running on 7860, change the port.
# app.demo.launch(
#     server_name="127.0.0.1",
#     server_port=7860,
#     share=False,
#     show_error=True,
#     css=app.CSS,
# )